# Exploratory Data Analysis (EDA) & Pemodelan Awal — Dataset Abalone
**Proyek UAS Komputasi Awan — Kelompok 2 (Topik 2: Model Prediktif sebagai Cloud API)**
- **File:** `src/analysis/eda.ipynb`
- **Tujuan:** Memahami struktur, distribusi, korelasi, dan pencilan (*outliers*) pada dataset Abalone, serta memverifikasi visualisasi hasil evaluasi model sebelum diserahkan ke API Layer.

## Bagian 0: Instalasi Library (Jalankan jika muncul `ModuleNotFoundError`)
Jika menjalankan di VS Code lokal dengan environment Python baru, jalankan sel ini satu kali untuk menginstal library yang dibutuhkan.

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn openpyxl joblib ipywidgets

## Bagian 1: Import Library & Pilih File Dataset Secara Manual
Sel ini akan langsung membuka jendela pemilihan file (*File Explorer* / dialog upload) sehingga Anda tinggal memilih file `Dataset_Abalone_Lengkap.xlsx`, `abalone.csv`, atau `abalone.data` dari folder di komputer Anda.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
os.makedirs("docs/evidence", exist_ok=True)

COLUMNS = [
    "Sex", "Length", "Diameter", "Height",
    "Whole weight", "Shucked weight", "Viscera weight", "Shell weight", "Rings"
]

def pilih_dan_muat_file():
    file_path = None
    # 1. Jika dijalankan di Google Colab
    try:
        from google.colab import files
        print("Silakan pilih file dataset melalui tombol Choose Files di bawah:")
        uploaded = files.upload()
        if uploaded:
            file_path = list(uploaded.keys())[0]
    except ImportError:
        # 2. Jika dijalankan di VS Code / Jupyter Lokal -> buka jendela File Explorer Windows/Mac
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        file_path = filedialog.askopenfilename(
            title="Pilih Dataset Abalone (.xlsx, .csv, atau .data)",
            filetypes=[
                ("Dataset Abalone", "*.xlsx *.xls *.csv *.data"),
                ("Semua File", "*.*")
            ]
        )
        root.destroy()

    if not file_path:
        raise FileNotFoundError("Tidak ada file yang dipilih. Jalankan ulang sel ini untuk memilih file.")

    print(f"File terpilih: {file_path}")
    if file_path.endswith((".xlsx", ".xls")):
        xls = pd.ExcelFile(file_path)
        sheet = "Dataset_Abalone" if "Dataset_Abalone" in xls.sheet_names else xls.sheet_names[0]
        return pd.read_excel(file_path, sheet_name=sheet)
    elif file_path.endswith(".csv"):
        return pd.read_csv(file_path)
    else:
        return pd.read_csv(file_path, header=None, names=COLUMNS)

df = pilih_dan_muat_file()
print(f"Ukuran dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
display(df.head())

## Bagian 2: Pemeriksaan Kualitas Data (Tipe Kolom, Missing Values, Duplikasi & Anomali)
Memastikan tidak ada nilai kosong, memeriksa duplikasi, dan mengecek nilai ekstrem seperti `Height == 0`.

In [ ]:
print("=== TIPE DATA & MISSING VALUES ===")
info_df = pd.DataFrame({
    "Tipe Data": df.dtypes,
    "Missing Values": df.isnull().sum(),
    "Unique Values": df.nunique()
})
display(info_df)

print(f"Jumlah baris duplikat: {df.duplicated().sum()}")
print(f"Jumlah observasi dengan Height == 0: {(df['Height'] == 0).sum()}")
display(df[df["Height"] == 0])

print("\n=== STATISTIK DESKRIPTIF NUMERIK ===")
display(df.describe().T)

## Bagian 3: Visualisasi Distribusi Target (`Rings`), Kategori `Sex`, dan Pencilan
Grafik disimpan ke folder `docs/evidence/` untuk dilampirkan pada `docs/analysis-report.md`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Histogram & KDE Target Rings
sns.histplot(df["Rings"], bins=28, kde=True, ax=axes[0], color="#1E3A5F")
axes[0].set_title("Distribusi Variabel Target (Rings)")
axes[0].set_xlabel("Jumlah Cincin (Rings)")
axes[0].set_ylabel("Frekuensi")

# 2. Boxplot Rings berdasarkan Kategori Sex
sns.boxplot(data=df, x="Sex", y="Rings", order=["M", "F", "I"], ax=axes[1], palette="Blues")
axes[1].set_title("Sebaran Rings berdasarkan Sex")
axes[1].set_xlabel("Jenis Kelamin (M=Male, F=Female, I=Infant)")
axes[1].set_ylabel("Rings")

# 3. Scatter Plot Shell weight vs Rings
sns.scatterplot(data=df, x="Shell weight", y="Rings", hue="Sex", alpha=0.5, ax=axes[2])
axes[2].set_title("Hubungan Shell Weight vs Rings")
axes[2].set_xlabel("Shell Weight (gram)")
axes[2].set_ylabel("Rings")

plt.tight_layout()
plt.savefig("docs/evidence/eda_distribusi_target.png", dpi=150)
plt.show()

## Bagian 4: Matriks Korelasi Antar Fitur Numerik
Melihat fitur fisik mana yang memiliki korelasi paling kuat dengan target `Rings` serta mendeteksi multikolinearitas antar ukuran fisik.

In [ ]:
plt.figure(figsize=(9, 7))
corr_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="YlGnBu", square=True)
plt.title("Matriks Korelasi Pearson Variabel Numerik Abalone")
plt.tight_layout()
plt.savefig("docs/evidence/eda_korelasi.png", dpi=150)
plt.show()

## Bagian 5: Visualisasi Evaluasi Model pada Data Uji (Aktual vs Prediksi & Residual)
Menjalankan evaluasi diagnostik setelah `train.py` menghasilkan `src/app/model/model.joblib`.

In [ ]:
import joblib
from sklearn.model_selection import train_test_split

if os.path.exists("src/app/model/model.joblib"):
    model = joblib.load("src/app/model/model.joblib")
    X = df[["Sex", "Length", "Diameter", "Height", "Whole weight", "Shucked weight", "Viscera weight", "Shell weight"]]
    y = df["Rings"]
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    y_pred = model.predict(X_test)
    residuals = y_test - y_pred

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Plot Aktual vs Prediksi
    axes[0].scatter(y_test, y_pred, alpha=0.45, color="#2C5282")
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
    axes[0].set_title("Nilai Aktual vs Prediksi (Test Set)")
    axes[0].set_xlabel("Rings Aktual")
    axes[0].set_ylabel("Rings Prediksi")

    # Plot Residual
    axes[1].scatter(y_pred, residuals, alpha=0.45, color="#2B6CB0")
    axes[1].axhline(0, color="r", linestyle="--", lw=2)
    axes[1].set_title("Plot Residual pada Data Uji")
    axes[1].set_xlabel("Rings Prediksi")
    axes[1].set_ylabel("Residual (Aktual - Prediksi)")

    plt.tight_layout()
    plt.savefig("docs/evidence/evaluasi_test_set.png", dpi=150)
    plt.show()
else:
    print("Jalankan src/analysis/train.py terlebih dahulu untuk menghasilkan model.joblib.")